# SIGN NAVをGoogle Colabで実行する

矢印標識の画像認識から左右輪の制御までをつなぐ、教育用の平面移動ロボットシミュレーションです。

**コードZIPを1つアップロードし、上から順に実行してください。** 保存済みの学習済み重みを使います。GPUと外部AI APIは不要です。ここで取得する標識画像は合成パネル画像で、実カメラの映像ではありません。

同梱コードはローカルのPython 3.12.14で動作確認しています。このノートブックはColab上での実行確認を行っていません。各スクリプトは別のPythonプロセスで動かします。

## 1. コードZIPをアップロードして展開

`main.py`、`vision.py`、`models/sign_mlp.npz` が入ったコードZIPを選んでください。新しい一時フォルダへ展開し、ZIP内の絶対パス、親ディレクトリへの移動、シンボリックリンクを除外します。

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("コードZIPを1つ選択してください。")
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
if not uploaded_name.lower().endswith(".zip"):
    raise ValueError("ZIP形式のコード一式を選択してください。")

EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix="sign_nav_", dir="/content")).resolve()
with zipfile.ZipFile(io.BytesIO(uploaded_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 5000:
        raise ValueError("ZIP内の項目数が多すぎます。コード一式のZIPを確認してください。")
    if sum(item.file_size for item in members) > 300 * 1024 * 1024:
        raise ValueError("展開後のサイズが300 MiBを超えます。コード一式のZIPを確認してください。")
    validated = []
    seen = set()
    for item in members:
        name = item.filename
        relative = PurePosixPath(name)
        mode = item.external_attr >> 16
        if (not name or relative.is_absolute() or ".." in relative.parts
                or "\\" in name or ":" in name or stat.S_ISLNK(mode)):
            raise ValueError("安全に展開できない項目が含まれています。")
        target = (EXTRACT_ROOT / Path(*relative.parts)).resolve()
        if not target.is_relative_to(EXTRACT_ROOT):
            raise ValueError("展開先の外を指す項目があります。")
        if target in seen:
            raise ValueError("ZIP内に重複するパスがあります。")
        seen.add(target)
        validated.append((item, target))
    for item, target in validated:
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)

candidates = [
    path.parent for path in EXTRACT_ROOT.rglob("main.py")
    if (path.parent / "vision.py").is_file()
    and (path.parent / "mission.py").is_file()
    and (path.parent / "requirements-core.txt").is_file()
    and (path.parent / "models/sign_mlp.npz").is_file()
]
if len(candidates) != 1:
    raise ValueError("SIGN NAVのコード一式を1つだけ含むZIPをアップロードしてください。")
PROJECT = candidates[0]
print("SIGN NAVのコードとモデルを確認しました。")
print("Python:", sys.version.split()[0])

## 2. 最小構成の依存関係をインストール

保存済みモデルを使う実行にはNumPyとPillowを使用します。

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-core.txt")],
    check=True,
)

def run_python(*arguments):
    task_environment = os.environ.copy()
    task_environment["OPENBLAS_NUM_THREADS"] = "1"
    task_environment["OMP_NUM_THREADS"] = "1"
    subprocess.run(
        [sys.executable, *arguments],
        cwd=PROJECT,
        env=task_environment,
        check=True,
    )

print("最小構成を準備しました。")

## 3. デモ・6条件の比較・記録の検証

標準条件の6条件で、学習済みMLP・常に直進の2方式を比較します。成功条件は「3つの標識すべてに従い、接触なくゴールへ到着すること」です。

In [ ]:
run_python("main.py")
run_python("experiments.py")
run_python("verify_navigation.py")

demo = json.loads((PROJECT / "results/demo/mission.json").read_text())
verification = json.loads((PROJECT / "results/navigation_verification.json").read_text())
print("デモの結果:", demo["result"])
print("検証:", verification["passed"], "/", verification["total"])

### デモで入力された標識画像

この画像が分類器へ渡された入力です。正解ラベルを分類器へ渡しているわけではありません。

In [ ]:
from IPython.display import display, Image as DisplayImage

for observation in demo["observations"]:
    prediction = observation["prediction"]
    print("分岐", observation["index"] + 1, "予測:", prediction["name"])
    display(DisplayImage(filename=str(PROJECT / "results/demo" / observation["image"]), width=192))

## 4. 任意：学習から再実行する

再学習する場合は、次のセルの `RUN_TRAINING = True` に変更して実行します。合成画像の学習・検証・標準テスト・強い変動テストを再生成し、画像分類の正解数を確認します。続けてデモ・走行比較・検証も更新します。

同梱の結果は、標準テスト300枚中300枚正解（100.00%）、強い変動テスト300枚中139枚正解（46.33%）です。学習用は900枚、学習途中の確認用は150枚です。

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
        check=True,
    )
    run_python("train.py")
    run_python("main.py")
    run_python("experiments.py")
    run_python("verify_navigation.py")
    metrics = json.loads((PROJECT / "results/vision_metrics.json").read_text())
    print(json.dumps(metrics["evaluations"], indent=2))
else:
    print("保存済みの学習済みモデルを利用します。")

## 5. 任意：動画を再生成する

動画を作る場合は `MAKE_VIDEO = True` に変更して実行します。FFmpegがない場合はこのセルで追加します。動画生成は画像を多数描画するため、デモ実行より時間がかかります。

同梱デモは82.10秒、1920×1080、20 fps、走行区間1倍速です。音声はなく、日本語字幕で説明します。

In [ ]:
MAKE_VIDEO = False
VIDEO_PATH = PROJECT / "deliverables/SIGN_NAV.mp4"

if MAKE_VIDEO:
    if shutil.which("ffmpeg") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
    run_python("make_video.py", "--out", str(VIDEO_PATH))
    from IPython.display import Video
    display(Video(filename=str(VIDEO_PATH), embed=True, width=960))
else:
    print("動画の再生成を省略しました。")

### 生成した動画をダウンロード

前のセルで動画を生成した場合に実行できます。

In [ ]:
if VIDEO_PATH.is_file():
    files.download(str(VIDEO_PATH))
else:
    print("動画をダウンロードする場合は、MAKE_VIDEO = Trueで前のセルを実行してください。")

## 実装と記録を読む

- `results/demo/mission.json`：デモの判断・イベント・結果
- `results/demo/telemetry.csv`：位置・左右輪・制御状態
- `results/benchmark/summary.json`：走行比較
- `results/navigation_verification.json`：10項目の検証
- `results/vision_metrics.json`：学習と独立画像評価
- `README.md`：実装範囲・再現方法
- `THIRD_PARTY_NOTICES.md`：使用ソフトウェアとフォントの出典

制作支援：設計案・コード・説明文の作成に生成AIを使用し、数値は同梱プログラムの実行結果を使用しています。